# WBGT Learning Module 1

### Learning goals

You will learn to:

1. identify every required input and unit,
2. calculate relative humidity from ERA5 temperature and dew point,
3. calculate interval-average solar geometry,
4. understand the black-globe and wetted-wick energy balances,
5. solve both temperatures iteratively,
6. combine them into outdoor WBGT,
7. compare explicit WBGT with sWBGT and ESI,
8. calculate ISO-style labor productivity.

The implementation follows the physical model discussed by Kong and Huber
(2022).

In [ ]:
#Imports

import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import wbgt_physics as wbgt

print("Using:", Path(wbgt.__file__).resolve())

## 2. WBGT definition

For outdoor conditions with solar radiation:

\[
WBGT = 0.7T_{nwb} + 0.2T_g + 0.1T_a
\]

- \(T_{nwb}\): natural wet-bulb temperature
- \(T_g\): black-globe temperature
- \(T_a\): air temperature

The two sensor temperatures cannot usually be obtained with one algebraic
step. Each is the root of an energy-balance equation.

## 3. Required ERA5-style inputs

| Input | Unit |
|---|---|
| 2 m air temperature | K |
| 2 m dew-point temperature | K |
| Surface pressure | Pa |
| 10 m wind components | m s\(^{-1}\) |
| Downward and upward shortwave radiation | W m\(^{-2}\) |
| Downward and upward longwave radiation | W m\(^{-2}\) |
| Direct fraction of downward shortwave | 0–1 |
| Sunlit-only interval-average cosine zenith angle | 0–1 |

ERA5 radiation accumulations must be converted to W m\(^{-2}\) before this
calculation.

## 4. Humidity calculation

Relative humidity is the ratio of actual vapor pressure to saturation vapor
pressure. ERA5 provides dew point, so the dew-point saturation vapor pressure
represents the actual vapor pressure.

In [ ]:
print(
    inspect.getsource(
        wbgt.relative_humidity_from_dewpoint
    )
)

## 5. Solar geometry

The direct-beam terms use the cosine of solar zenith angle. For interval
radiation, we need an interval-average cosine.

Two values are useful:

- a complete-interval average for detecting night and low-sun conditions,
- a sunlit-only average for projecting direct radiation without creating
  artificial sunrise and sunset spikes.

In [ ]:
print(
    inspect.getsource(
        wbgt.solar_cosines
    )
)

## 6. Air properties, wind conversion, and convection

The explicit model accounts for heat transfer around a globe and a wetted
cylindrical wick. ERA5 wind is measured at 10 m, so the model converts it to
2 m using radiation- and wind-dependent stability classes.

In [ ]:
for function in [
    wbgt.wind_2m_from_10m,
    wbgt.sphere_convection_coefficient,
    wbgt.cylinder_convection_coefficient,
]:
    print(inspect.getsource(function))
    print("-" * 70)

In [ ]:
# Numerical root solver

print(
    inspect.getsource(
        wbgt._bisection_array
    )
)

## 8. Globe and natural wet-bulb solvers

In [ ]:
print(
    inspect.getsource(
        wbgt.globe_temperature_k
    )
)

In [ ]:
print(
    inspect.getsource(
        wbgt.natural_wet_bulb_temperature_k
    )
)

## 9. Build an ERA5-like case

These are synthetic values, not an observation.

In [ ]:
case = {
    "time_utc": np.datetime64("2024-03-21T12:00"),
    "latitude_deg": 5.6,
    "longitude_deg": -0.2,
    "air_temperature_k": 306.15,
    "dewpoint_temperature_k": 298.15,
    "surface_pressure_pa": 100_500.0,
    "u10_ms": 1.8,
    "v10_ms": 0.9,
    "shortwave_down_wm2": 750.0,
    "shortwave_up_wm2": 135.0,
    "longwave_down_wm2": 410.0,
    "longwave_up_wm2": 470.0,
    "direct_shortwave_wm2": 500.0,
    "interval_hours": 1.0,
}

pd.Series(case)

## 10. Derive humidity, wind speed, direct fraction, and solar cosines

In [ ]:
relative_humidity_pct = float(
    wbgt.relative_humidity_from_dewpoint(
        case["air_temperature_k"],
        case["dewpoint_temperature_k"],
        case["surface_pressure_pa"],
    )
)

wind_10m_ms = float(
    np.hypot(
        case["u10_ms"],
        case["v10_ms"],
    )
)

direct_fraction = (
    case["direct_shortwave_wm2"]
    / case["shortwave_down_wm2"]
)
direct_fraction = float(
    np.clip(
        direct_fraction,
        0.0,
        0.9,
    )
)

full_cosine, sunlit_cosine = wbgt.solar_cosines(
    np.array([case["time_utc"]]),
    np.array([case["latitude_deg"]]),
    np.array([case["longitude_deg"]]),
    case["interval_hours"],
)

derived = pd.Series({
    "Relative humidity (%)": relative_humidity_pct,
    "10 m wind speed (m/s)": wind_10m_ms,
    "Direct fraction": direct_fraction,
    "Full-interval cosine": float(full_cosine.squeeze()),
    "Sunlit-only cosine": float(sunlit_cosine.squeeze()),
})

derived.round(4)

## 11. Calculate the explicit sensor temperatures and WBGT

In [ ]:
result = wbgt.calculate_wbgt(
    air_temperature_k=case["air_temperature_k"],
    relative_humidity_pct=relative_humidity_pct,
    pressure_pa=case["surface_pressure_pa"],
    wind_ms=wind_10m_ms,
    shortwave_down_wm2=case["shortwave_down_wm2"],
    shortwave_up_wm2=case["shortwave_up_wm2"],
    longwave_down_wm2=case["longwave_down_wm2"],
    longwave_up_wm2=case["longwave_up_wm2"],
    direct_fraction=direct_fraction,
    sunlit_cosine=float(sunlit_cosine.squeeze()),
    wind_height_m=10,
)

result_table = pd.Series({
    "Air temperature (°C)": (
        case["air_temperature_k"] - 273.15
    ),
    "Natural wet-bulb temperature (°C)": float(
        result.natural_wet_bulb_c
    ),
    "Globe temperature (°C)": float(
        result.globe_temperature_c
    ),
    "Outdoor WBGT (°C)": float(
        result.wbgt_c
    ),
    "Converted 2 m wind (m/s)": float(
        result.wind_2m_ms
    ),
})

result_table.round(2)

### Check the weighted sum

In [ ]:
weighted_sum_c = (
    0.7 * float(result.natural_wet_bulb_c)
    + 0.2 * float(result.globe_temperature_c)
    + 0.1 * (
        case["air_temperature_k"] - 273.15
    )
)

print("Returned WBGT:", float(result.wbgt_c))
print("Weighted sum: ", weighted_sum_c)
print("Difference:   ", float(result.wbgt_c) - weighted_sum_c)

## 12. Compare with sWBGT and ESI

In [ ]:
air_temperature_c = (
    case["air_temperature_k"] - 273.15
)

comparison = pd.DataFrame({
    "Metric": [
        "Explicit WBGT",
        "sWBGT",
        "ESI",
    ],
    "Value (°C)": [
        float(result.wbgt_c),
        float(
            wbgt.simplified_wbgt_c(
                air_temperature_c,
                relative_humidity_pct,
            )
        ),
        float(
            wbgt.environmental_stress_index_c(
                air_temperature_c,
                relative_humidity_pct,
                case["shortwave_down_wm2"],
            )
        ),
    ],
})

comparison["Difference from explicit (°C)"] = (
    comparison["Value (°C)"]
    - float(result.wbgt_c)
)

comparison.round(2)

## 13. Sensitivity experiments

In [ ]:
def calculate_case(
    air_temperature_c=33.0,
    relative_humidity_pct=65.0,
    wind_10m_ms=2.0,
    shortwave_down_wm2=750.0,
):
    shortwave_up_wm2 = 0.18 * shortwave_down_wm2
    direct_fraction = (
        0.0
        if shortwave_down_wm2 == 0.0
        else 0.65
    )

    answer = wbgt.calculate_wbgt(
        air_temperature_k=air_temperature_c + 273.15,
        relative_humidity_pct=relative_humidity_pct,
        pressure_pa=100_500.0,
        wind_ms=wind_10m_ms,
        shortwave_down_wm2=shortwave_down_wm2,
        shortwave_up_wm2=shortwave_up_wm2,
        longwave_down_wm2=410.0,
        longwave_up_wm2=470.0,
        direct_fraction=direct_fraction,
        sunlit_cosine=(
            1.0
            if shortwave_down_wm2 == 0.0
            else 0.85
        ),
        wind_height_m=10,
    )

    return float(answer.wbgt_c)


experiments = {
    "Air temperature (°C)": np.arange(28.0, 41.0, 2.0),
    "Relative humidity (%)": np.arange(30.0, 91.0, 10.0),
    "10 m wind speed (m/s)": np.array([0.5, 1.0, 2.0, 3.0, 5.0]),
    "Shortwave radiation (W/m²)": np.array([0.0, 300.0, 500.0, 700.0, 900.0]),
}

results = {}

for variable, values in experiments.items():
    rows = []

    for value in values:
        arguments = {}

        if variable == "Air temperature (°C)":
            arguments["air_temperature_c"] = value
        elif variable == "Relative humidity (%)":
            arguments["relative_humidity_pct"] = value
        elif variable == "10 m wind speed (m/s)":
            arguments["wind_10m_ms"] = value
        else:
            arguments["shortwave_down_wm2"] = value

        rows.append(
            {
                "Input": value,
                "WBGT (°C)": calculate_case(**arguments),
            }
        )

    results[variable] = pd.DataFrame(rows)

results["Air temperature (°C)"].round(2)

In [ ]:
for x_label, table in results.items():
    plt.figure(figsize=(7, 4))
    plt.plot(
        table["Input"],
        table["WBGT (°C)"],
        marker="o",
    )
    plt.xlabel(x_label)
    plt.ylabel("Explicit outdoor WBGT (°C)")
    plt.title(f"Sensitivity to {x_label}")
    plt.grid(True, alpha=0.3)
    plt.show()

## 14. ISO-style labor productivity

In [ ]:
metabolic_rates_w = np.array(
    [200.0, 300.0, 400.0, 600.0]
)

labor_table = pd.DataFrame({
    "Metabolic rate (W)": metabolic_rates_w,
    "WBGT limit (°C)": wbgt.iso_wbgt_limit_c(
        metabolic_rates_w
    ),
    "Productivity (%)": (
        100.0
        * wbgt.iso_labor_productivity(
            float(result.wbgt_c),
            metabolic_rates_w,
        )
    ),
})

labor_table.round(2)

## Exercises

1. Change the dew point from 25°C to 27°C.
2. Reduce the wind speed to 0.5 m s\(^{-1}\).
3. Compare a sunny and nighttime case.
4. Change the root tolerance from 0.01 K to 0.001 K and compare the result.
5. Compare labor productivity for 200 W and 600 W.

## References

- Kong, Q., & Huber, M. (2022). *Explicit calculations of wet-bulb globe temperature compared with approximations and why it matters for labor productivity*. Earth's Future, 10, e2021EF002334.
- Liljegren, J. C., et al. (2008). *Modeling the Wet Bulb Globe Temperature Using Standard Meteorological Measurements*. Journal of Occupational and Environmental Hygiene, 5, 645–655.